# Libraries

In [ ]:
from modules import checks_formatter, query_tools, utils

# Main

In [ ]:
BQ_PROJECT = "your-gcp-project-id"
BQ_DATASET = "monitoring_dataset"
CONTROL_SPREADSHEET_ID = "spreadsheet-id-containing-check-definitions"

In [ ]:
SOURCE_VIEWS_FILE_PATH = "queries/views/create_source_views.sql"
CHECKS_VIEWS_FILE_PATH = "queries/views/create_check_view.sql"
DELETE_EXISTING_VIEWS_FILE_PATH = "queries/views/delete_existing_views.sql"

## Initialise formatter

In [ ]:
formatter = (
    checks_formatter
    .ChecksFormatter(
        control_spreadsheet_id=CONTROL_SPREADSHEET_ID,
        monitoring_bq_project=BQ_PROJECT,
        monitoring_bq_dataset=BQ_DATASET
    )
)

## Format monitoring views

This step reads check definitions from a control spreadsheet and formats SQL view creation statements.

In [ ]:
formatter.format_views()

# Create View Files

In [ ]:
with open(SOURCE_VIEWS_FILE_PATH, "w") as source_queries:
    source_queries.write(formatter.create_source_views_query)

with open(CHECKS_VIEWS_FILE_PATH, "w") as checks_query:
    checks_query.write(formatter.create_checks_views_query)

# Delete Existing Views

In [ ]:
existing_views_delete_statements = (
    query_tools
    .run_query(
        BQ_PROJECT,
        "utils/get_delete_views_statements.sql",
        params={
            "PARAM-BQ-PROJECT": BQ_PROJECT,
            "PARAM-BQ-DATASET": BQ_DATASET
        }
    )
)

delete_views_query = (
    "\n"
    .join(
        utils.flatten_array(
            existing_views_delete_statements.values
        )
    )
)

with open(DELETE_EXISTING_VIEWS_FILE_PATH, "w") as delete_views:
    delete_views.write(delete_views_query)

# Run View Creation Queries

In [ ]:
delete_existing_views_response = (
    query_tools
    .run_query(
        BQ_PROJECT,
        "views/delete_existing_views.sql"
    )
)

In [ ]:
source_views_creation_response = (
    query_tools
    .run_query(
        BQ_PROJECT,
        "views/create_source_views.sql"
    )
)

In [ ]:
checks_view_creation_response = (
    query_tools
    .run_query(
        BQ_PROJECT,
        "views/create_check_view.sql"
    )
)